In [1]:

# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"


# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# policy
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.envs.utils import env_to_policy_features

# record utils
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import _init_rerun
from lerobot.record import record_loop

# torch
import torch

# utils
from dotenv import load_dotenv
import numpy as np

# my code
from src.utils import scroll_print
from src.const import TABLE_START_POSE
from src.paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR, EVAL_DIR, OUTPUTS_DIR
from envs.so101_env_utils import SO101TASKS, SO101_LEADER_JOINT_RANGE

# my env
from envs.so101_env_config import SO101EnvConfig, make_so101_env

# set up os_env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

%load_ext autoreload
%autoreload 2

C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


### Params:

Optional tasks:

In [2]:
print(SO101TASKS)

['TableLegAssembleTask', 'TableLegMoveTask']


In [3]:
# Data recording params
PUSH_TO_HUB     = False
SAVE_TO_DATASET = False

# Rollout pararms
NUM_EPISODES = 1
EPISODE_TIME_SEC = 60
EVAL_TYPE = 'sim'

# Policy checkpoint params
REPO_NAME         = 'so101_table_leg_move'
EXPERIMENT_NAME   = '50_episodes_no_augmentations'
POLICY_CHECKPOINT = '080000'
FPS = 30
POLICY_TYPE = 'act'

Log to hub if pushing:

In [4]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

### Init Policy:

In [5]:
# Initialize the policy
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / POLICY_CHECKPOINT / 'pretrained_model'
policy = ACTPolicy.from_pretrained(pretrained_name_or_path = policy_path, local_files_only=True)
policy.eval()
scroll_print(policy.config)

HFValidationError: Repo id must use alphanumeric chars or '-', '_', '.', '--' and '..' are forbidden, '-' and '.' cannot start or end the name, max length is 96: 'C:\Github\IDC_Project-Sim_Real_Sim\policies\act\so101_table_leg_move\50_episodes_no_augmentations\checkpoints\080000\pretrained_model'.

### Build Env:

In [ ]:
env_cfg = SO101EnvConfig(
    task                  = "TableLegAssembleTask",
    obs_type              = "pixels_agent_pos",
    device                = device,
    external_joint_ranges = SO101_LEADER_JOINT_RANGE,
    control_time_s        = 100,
    joint_init_pose       = np.array(list(TABLE_START_POSE.values()), dtype = np.float32)
    )
env = make_so101_env(env_cfg, 
                    torch_actions = True,
                    lerobot_obs   = True)
scroll_print(env_cfg)

### Init Dataset:

In [ ]:
# configure the dataset features
if SAVE_TO_DATASET:
    action_features = hw_to_dataset_features(robot.action_features, "action")
    obs_features = hw_to_dataset_features(robot.observation_features, "observation")
    dataset_features = {**action_features, **obs_features}

    # create dataset
    policy_eval_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"

    kwargs = {
    "repo_id": f"{HF_NAME}/{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    "root"   : str(policy_eval_path)
    }

    if policy_eval_path.exists():
        dataset=LeRobotDataset(**kwargs)
    else:
        dataset = LeRobotDataset.create(
            **kwargs,
            fps=FPS,
            features=dataset_features,
            robot_type=robot.name,
            use_videos=True,
            image_writer_threads=4,
        )
        

### Rollouts

In [ ]:
from src.env_rollout import env_rollout
env_rollout(
    display_rerun = True,
    env_cfg       = env_cfg,
    env           = env,
    num_episodes  = NUM_EPISODES,
    policy        = policy,
    dataset       = None,
)

Left arrow key pressed. Exiting loop and rerecord the last episode...
Episode 1/1: reward=0.00, success=False


{'episodes': [{'reward': 0.0, 'done': False, 'success': False}],
 'avg_reward': np.float64(0.0),
 'success_rate': 0.0}